In [ ]:
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

session = requests.Session()

# ==============================================
# STEP 1 - Ambil homepage
# ==============================================

home_url = "https://21cineplex.com"
home_response = session.get(home_url)
home_html = home_response.text
home_soup = BeautifulSoup(home_html, "html.parser")

# ==============================================
# STEP 2 - Ambil token AJAX
# ==============================================

token = re.search(r'token:\s*"([^"]+)"', home_html).group(1)

print("Token ditemukan")

# ==============================================
# STEP 3 - Ambil semua kota
# ==============================================

city_dropdown = home_soup.find("select", id="cityChanged")

cities = []

for city in city_dropdown.find_all("option"):

    cities.append({
        "city_id": city["value"],
        "city_name": city.text.strip()
    })

cities_df = pd.DataFrame(cities)

print(f"Total kota: {len(cities_df)}")

# ==============================================
# STEP 4 - Loop semua kota
# ==============================================

all_movies = []

for city in cities:

    print(f"Processing: {city['city_name']}")

    payload = {
        "data": "getMovNowplaying",
        "city_id": city["city_id"],
        "token": token
    }

    response = session.post(
        "https://21cineplex.com/script/ajax-home-nowplaying.php",
        data=payload,
        headers={
            "Referer": "https://21cineplex.com/",
            "X-Requested-With": "XMLHttpRequest"
        }
    )

    try:
        ajax_data = response.json()

    except Exception:

        print(f"Gagal di {city['city_name']}")

        continue

    ajax_soup = BeautifulSoup(ajax_data["msg"], "html.parser")

    movie_cards = ajax_soup.find_all("div", class_="movie")

    for movie in movie_cards:

        try:

            title = movie.find("h4").text.strip()

            link = movie.find("a")["href"]

            all_movies.append({
                "city_id": city["city_id"],
                "city_name": city["city_name"],
                "title": title,
                "link": link
            })

        except Exception:
            pass

    time.sleep(
        random.uniform(0.5, 1.5)
    )

# ==============================================
# STEP 5 - Hasil mentah
# ==============================================

movies_df = pd.DataFrame(all_movies)

print(movies_df.shape)